# “Doc-level routing” bằng centroid

In [1]:
import json, numpy as np
from collections import defaultdict

CHUNKS_PATH = r"D:\GitHub\ChatBot\output_nghidinh\chunks_clean_norm.json"
EMB_PATH    = r"D:\GitHub\ChatBot\vector_data\legal_hf_cosine\embeddings.npy"
OUT_PATH    = r"D:\GitHub\ChatBot\vector_data\legal_hf_cosine\doc_centroids.npz"

chunks = json.load(open(CHUNKS_PATH, "r", encoding="utf-8"))
if isinstance(chunks, dict) and "chunks" in chunks:
    chunks = chunks["chunks"]

emb = np.load(EMB_PATH).astype("float32")

groups = defaultdict(list)
for i, c in enumerate(chunks):
    vb = c.get("metadata", {}).get("van_ban", "UNKNOWN")
    groups[vb].append(i)

doc_names = []
centroids = []
doc_ids = []  # list of list ids, saved separately

for vb, idxs in groups.items():
    vec = emb[idxs].mean(axis=0)
    # normalize centroid để cosine = dot
    vec = vec / (np.linalg.norm(vec) + 1e-12)
    doc_names.append(vb)
    centroids.append(vec)
    doc_ids.append(np.array(idxs, dtype=np.int32))

centroids = np.stack(centroids, axis=0).astype("float32")

# save
np.savez_compressed(OUT_PATH, doc_names=np.array(doc_names), centroids=centroids)
# save mapping ids separate (json) cho dễ load
json.dump({doc_names[i]: doc_ids[i].tolist() for i in range(len(doc_names))},
          open(OUT_PATH.replace(".npz", "_ids.json"), "w", encoding="utf-8"),
          ensure_ascii=False)

print("Saved:", OUT_PATH, "docs=", len(doc_names))

Saved: D:\GitHub\ChatBot\vector_data\legal_hf_cosine\doc_centroids.npz docs= 23


In [2]:
import json, numpy as np
from sentence_transformers import SentenceTransformer

CHUNKS_PATH = r"D:\GitHub\ChatBot\output_nghidinh\chunks_clean_norm.json"
EMB_PATH    = r"D:\GitHub\ChatBot\vector_data\legal_hf_cosine\embeddings.npy"
DOC_NPZ     = r"D:\GitHub\ChatBot\vector_data\legal_hf_cosine\doc_centroids.npz"
DOC_IDS     = r"D:\GitHub\ChatBot\vector_data\legal_hf_cosine\doc_centroids_ids.json"
MODEL_NAME  = "Quockhanh05/Vietnam_legal_embeddings"
DEVICE      = "cpu"

chunks = json.load(open(CHUNKS_PATH, "r", encoding="utf-8"))
if isinstance(chunks, dict) and "chunks" in chunks:
    chunks = chunks["chunks"]

emb = np.load(EMB_PATH).astype("float32")

npz = np.load(DOC_NPZ, allow_pickle=True)
doc_names = npz["doc_names"]
centroids = npz["centroids"].astype("float32")

doc2ids = json.load(open(DOC_IDS, "r", encoding="utf-8"))

encoder = SentenceTransformer(MODEL_NAME, device=DEVICE)

def retrieve_hier(question, top_docs=2, topk=5):
    q = encoder.encode([question], normalize_embeddings=True).astype("float32")[0]

    # 1) chọn top docs bằng cosine
    doc_scores = centroids @ q
    topd = np.argsort(-doc_scores)[:top_docs]

    candidate_ids = []
    print("Top docs:")
    for j in topd:
        name = str(doc_names[j])
        sc = float(doc_scores[j])
        print(f" - {sc:.4f} | {name[:120]}")
        candidate_ids.extend(doc2ids[name])

    # 2) rank chunks trong candidate_ids
    sub = emb[candidate_ids]
    scores = sub @ q
    top = np.argsort(-scores)[:topk]

    print("="*90)
    print("QUESTION:", question)
    print("="*90)
    for rank, t in enumerate(top, 1):
        cid = candidate_ids[int(t)]
        c = chunks[cid]
        m = c.get("metadata", {})
        print(f"[{rank}] sc={float(scores[int(t)]):.4f} | {m.get('van_ban','')[:90]} | Điều {m.get('dieu','')} | Khoản {m.get('khoan','-')} | id={cid}")
        print("   ", c.get("text","").replace("\n"," ")[:220], "...")
        print("-"*90)

if __name__ == "__main__":
    retrieve_hier("Cấp xã được giao nhiệm vụ gì về lệ phí trước bạ?", top_docs=3, topk=5)

c:\Users\ADMIN\.conda\envs\ocr311\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.6 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "



Top docs:
 - 0.4477 | NGHỊ ĐỊNH Quy định về phân định thẩm quyền của chính quyền địa phương 02 cấp trong lĩnh vực quản lý nhà nước của bộ tài 
 - 0.4179 | NGHỊ ĐỊNH Quy định về phân định thẩm quyền của chính quyền địa phương 02 cấp trong lĩnh vực quản lý nhà nước của Bộ Y tế
 - 0.3645 | NGHỊ ĐỊNH "Quy định về phân định thẩm quyền của chính quyền địa phương 02 cấp trong lĩnh vực quản lý nhà nước của Bộ Nội
QUESTION: Cấp xã được giao nhiệm vụ gì về lệ phí trước bạ?
[1] sc=0.5747 | NGHỊ ĐỊNH Quy định về phân định thẩm quyền của chính quyền địa phương 02 cấp trong lĩnh vự | Điều 10 | Khoản 1 | id=1655
    Tại điểm b Khoản 1 Điều 10 Trước ngày 25 hằng tháng, Chủ tịch Ủy ban nhân dân cấp xã căn cứ danh sách đối tượng thụ hưởng (bao gồm đối tượng tăng, giảm; đối tượng hưởng một lần); số kinh phí chi trả tháng sau (bao gồm cả ...
------------------------------------------------------------------------------------------
[2] sc=0.5608 | NGHỊ ĐỊNH Quy định về phân định thẩm quyền của chính quyền